# Análise Exploratória de Dados (EDA) - Propensão de Compra
## Tech Challenge - Fase 2 (FIAP)

Este notebook realiza a análise exploratória do **Online Shoppers Purchasing Intention Dataset** (UCI), identificando os principais padrões de navegação e fatores determinantes para a conversão de compra (`Revenue`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Carga dos Dados e Visão Geral

In [ ]:
data_path = '../data/raw/online_shoppers_intention.csv'
df = pd.read_csv(data_path)
print(f'Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas')
df.head()

In [ ]:
df.info()

### 2. Análise da Variável Alvo (`Revenue`)
Avaliação do desbalanceamento de classes nas sessões do e-commerce.

In [ ]:
target_counts = df['Revenue'].value_counts(normalize=True) * 100
print('Distribuição percentual da variável alvo:')
print(target_counts)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Revenue', palette='viridis')
plt.title('Distribuição do Alvo (Revenue)')
plt.xlabel('Houve Compra? (Revenue)')
plt.ylabel('Quantidade de Sessões')
plt.show()

> **Insight:** A base apresenta forte desbalanceamento (~84.5% sem compra vs ~15.5% com compra). Por esse motivo, métricas como ROC-AUC, Recall e F1-Score são cruciais, e o modelo deve ser treinado com balanceamento de pesos (`class_weight='balanced'`).

### 3. Análise da Variável Mais Determinante: `PageValues`

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Revenue', y='PageValues', palette='Blues')
plt.title('PageValues por Status de Compra (Revenue)')
plt.yscale('log')
plt.show()

### 4. Taxas de Rejeição e Saída (`BounceRates` vs `ExitRates`)

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='BounceRates', y='ExitRates', hue='Revenue', alpha=0.6, palette='coolwarm')
plt.title('Bounce Rates vs Exit Rates segregados por Compra')
plt.show()

### 5. Sazonalidade: Conversões por Mês

In [ ]:
plt.figure(figsize=(10, 5))
order = ['Feb', 'Mar', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
sns.countplot(data=df, x='Month', hue='Revenue', order=order, palette='magma')
plt.title('Volume de Sessões e Compras por Mês')
plt.show()

### 6. Conclusões e Decisões para Modelagem
1. **Pré-processamento:** Normalização/Padronização com `StandardScaler` para numéricas contínuas e codificação categórica com `OneHotEncoder`.
2. **Desbalanceamento:** Uso de ponderação de classes nos estimadores (ex.: Random Forest com `class_weight='balanced'`).
3. **Feature Significance:** `PageValues`, `ExitRates` e `ProductRelated_Duration` são preditores fortemente correlacionados com o fechamento do carrinho.
4. **Métricas de Sucesso:** ROC-AUC (> 0.90) e Recall (> 0.80) para maximizar a captura de compradores reais sem penalizar excessivamente a precisão.